# Impedance ML Engineering — Session 02
## Build the first patient-safe ML baseline
**Core: 60 minutes · Classical ML · Optional extension: 20–40 minutes**

## Why this matters
Can a simple ridge model improve on persistence for next-visit telemetry in unseen patients?

Build the judgment to implement, review, and challenge data and modeling workflows in an ML engineering role.

## Learning objectives
- Define current-time inputs and future outcomes.
- Allocate patients to train, validation, and test before preprocessing.
- Build a mixed-type scikit-learn pipeline.
- Compare validation MAE with a persistence baseline.

## R-to-Python / prerequisite bridge
R resampling groups → `GroupShuffleSplit`; recipes → `ColumnTransformer` + `Pipeline`; penalized regression → `Ridge`; use splitter positions with `.iloc`.

## Suggested pacing
Tasks 1–3: 18 min · Tasks 4–6: 23 min · Tasks 7–8: 11 min · Retrieval/reasoning/reflection: 8 min. Task timings are approximate; if a foundational operation needs repetition, pause there and continue later. Extensions are outside the core hour.

## Inputs and analysis contract
Start Jupyter from the repository root and select **Python (learning-python)**. Read `data/ci_telemetry/README.md` for the complete dictionary. All inputs are local, fictional teaching data: 96 patients, one implant each, 360 visits, 144 directed cells per visit. Matrix rows stimulate; columns record; contact indices are 1–12. `voltage_mv / stim_current_ua` is a synthetic kΩ V/I proxy, not a confirmed manufacturer clinical impedance algorithm. QC-valid means `ok` AND finite voltage. Mask invalid cells; unknown does not mean measured zero. No hearing, disease, or device-failure labels are available.

Write explanations in new markdown cells or print them. Exercise cells are intentionally empty. No downloads or GPU are required.

## Concept and API reminders
Supervised regression learns a mapping from inputs X to a continuous outcome y. A baseline specifies what a useful model must improve upon. Persistence predicts the future from the current value without fitting parameters. Ridge minimizes squared residuals plus a penalty on coefficient size; scaling makes the penalty comparable across numeric features.

A `Pipeline` fits each transformation on the training data it receives; a `ColumnTransformer` applies separate recipes to numeric and categorical columns. Its structure is `ColumnTransformer([(name, transformer, columns), ...])`. CV/holdout splitters return integer positions, so use `.iloc`. MAE averages absolute residuals and retains outcome units. A negative sklearn MAE score is a maximization convention, not a negative physical error.

## Task 1 — Define the prediction unit (5 min)
Read `prediction_pairs.csv` directly as `pairs`. Acceptance: 264 rows, 96 patients, one row per current visit. Inspect current and target visit/month fields. Predict why a random row split can leak across overlapping pairs. The outcome is `target_diag_mean_kohm`, not a current-visit summary.

## Task 2 — Freeze available inputs (5 min)
Set numeric features to `diag_mean_kohm`, `diag_sd_kohm`, `near_mean_kohm`, `far_mean_kohm`, `n_diag_valid`, `age_at_implant_years`, `months_since_implant`, `planned_horizon_months`. Categories: `array_type`, `protocol`, `site`. Create X (11 columns), y, and patient groups. Exclude IDs, all `target_*` audit columns, and exact future timing. State when each predictor is known.

## Task 3 — Reserve test patients and validation patients (8 min)
Use `GroupShuffleSplit(n_splits=1,test_size=0.25,random_state=42)` to get development/test positions. Split development again with test_size=0.25 and seed 43, using development groups. Convert the second positions back to full-table positions. Acceptance: 54 train, 18 validation, 24 test patients; 146/51/67 rows; no patient overlap. Structural inspection of test IDs is allowed; outcomes stay unopened until Session 20. Hint: second indices index the development array, not the full table.

## Task 4 — Score persistence on validation (5 min)
Predict the next value as the current `diag_mean_kohm` for validation rows. Report MAE (lower is better, kΩ) and errors by planned horizon. API: `mean_absolute_error`. Also compute a training-mean `DummyRegressor` baseline trained on training outcomes. Explain why persistence is a stronger benchmark than a global mean.

## Task 5 — Build preprocessing inside the model (10 min)
Import `Pipeline`, `ColumnTransformer`, `SimpleImputer`, `StandardScaler`, `OneHotEncoder`, and `Ridge`. Numeric branch: median imputer then scaler. Categorical branch: most-frequent imputer then `OneHotEncoder(handle_unknown="ignore",sparse_output=False)`. Combine branches, append `Ridge(alpha=1.0)` as model. Fit only training rows. Hint: pipeline steps are `(name, object)` tuples; column-transformer branches are `(name, transformer, column_list)` tuples.

## Task 6 — Audit transformed shapes (5 min)
Inspect learned numeric medians and categorical categories through `named_steps` and `named_transformers_`. Call `get_feature_names_out` on the fitted preprocessor. Predict the relationship between 11 raw columns and encoded width before transforming validation. Acceptance: no NaN in transformed inputs; same encoded width for train and validation. Explain why a scaler fitted before splitting is leakage.

## Task 7 — Compare validation predictions (7 min)
Predict validation outcomes. Report ridge and persistence MAE on identical rows, plus model-minus-persistence MAE. Plot observed versus predicted with an identity line and equal axis limits. Calculate patient-average MAE by averaging errors within patient before averaging patients. The model need not win; justify the difference between patient and pair weighting.

## Task 8 — Record a reproducible baseline (7 min)
Write the target, feature list, split seeds, train/validation sizes, metrics, and next experiment. Save validation predictions under `artifacts/02/` with IDs for audit. Do not save or evaluate test predictions yet. Explain why repeated decisions on the same validation set make it exploratory rather than an independent final estimate.

## Retrieval practice
Audit `StandardScaler().fit_transform(all_rows)` followed by `train_test_split(...)`: identify preprocessing leakage and patient overlap, then describe both repairs.

## Professional reasoning
1. Why does excluding patient ID not eliminate patient leakage?
2. What could change when applying this benchmark to patients without observed follow-up?
3. Why does lower telemetry prediction error not establish a clinical benefit?

## Self-check and reflection
Record confidence **1–5**, hardest point, one thing now explainable, what needs repetition, and workload (too light / about right / too heavy). Restart the kernel and run your attempted cells in order. Ask for review when ready; generation does not count as completion.

## Optional extension — outside the core hour
Try ridge alphas 0.1, 1, 10 on training only and compare validation. Keep the experiment small; grouped CV becomes the selection tool in Session 06.

## API references
[scikit-learn grouped cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-iterators-for-grouped-data)